In [1]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

# Check that patches exist
train_imgs = glob.glob("dataset/processed/train/images/*.png")
test_imgs = glob.glob("dataset/processed/test/images/*.png")
print(f"Found {len(train_imgs)} train patches and {len(test_imgs)} test patches.")

PyTorch Version: 2.5.1
Active Device: cpu
Found 933 train patches and 125 test patches.


In [2]:
class NucleiSegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.image_paths = sorted(glob.glob(os.path.join(images_dir, "*.png")))
        self.mask_paths = sorted(glob.glob(os.path.join(masks_dir, "*.png")))
        assert len(self.image_paths) == len(self.mask_paths), "Mismatch between images and masks count!"

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 1. Load image (RGB) and normalize to [0, 1]
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))  # (H, W, C) -> (C, H, W)

        # 2. Load mask (Grayscale) and binarize
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE).astype(np.float32)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=0)  # (H, W) -> (1, H, W)

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

train_dataset = NucleiSegmentationDataset("dataset/processed/train/images", "dataset/processed/train/masks")
val_dataset = NucleiSegmentationDataset("dataset/processed/test/images", "dataset/processed/test/masks")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)

In [3]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.inc = DoubleConv(in_channels, 32)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))

        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(256, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(128, 64)
        self.up4 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(64, 32)

        self.outc = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5)
        x = self.conv_up1(torch.cat([x, x4], dim=1))
        x = self.up2(x)
        x = self.conv_up2(torch.cat([x, x3], dim=1))
        x = self.up3(x)
        x = self.conv_up3(torch.cat([x, x2], dim=1))
        x = self.up4(x)
        x = self.conv_up4(torch.cat([x, x1], dim=1))
        return self.outc(x)

class BCEDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits).view(-1)
        targets_flat = targets.view(-1)
        intersection = (probs * targets_flat).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets_flat.sum() + self.smooth)
        return bce_loss + (1.0 - dice)

model = UNet(in_channels=3, out_channels=1).to(device)
criterion = BCEDiceLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [4]:
os.makedirs("models", exist_ok=True)
num_epochs = 3  # Start with 3 epochs
best_val_loss = float("inf")

print("Starting U-Net Training...")
for epoch in range(1, num_epochs + 1):
    model.train()
    running_train_loss = 0.0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * images.size(0)

    train_loss = running_train_loss / len(train_loader.dataset)

    # Validation
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            running_val_loss += loss.item() * images.size(0)

    val_loss = running_val_loss / len(val_loader.dataset)
    print(f"Epoch [{epoch:02d}/{num_epochs:02d}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "models/unet.pth")
        print(f"  -> Checkpoint saved to models/unet.pth (Val Loss: {val_loss:.4f})")

print("\nTraining completed!")

Starting U-Net Training...
Epoch [01/03] | Train Loss: 0.7724 | Val Loss: 0.8076
  -> Checkpoint saved to models/unet.pth (Val Loss: 0.8076)
Epoch [02/03] | Train Loss: 0.5931 | Val Loss: 0.6640
  -> Checkpoint saved to models/unet.pth (Val Loss: 0.6640)
Epoch [03/03] | Train Loss: 0.5519 | Val Loss: 2.6377

Training completed!


In [ ]:
import time

num_epochs = 3  # Kept small for CPU training
best_val_loss = float("inf")

print(f"Starting training on {device} for {num_epochs} epochs...")
print("=" * 60)

for epoch in range(1, num_epochs + 1):
    start_time = time.time()
    
    # ------------------
    # Training Phase
    # ------------------
    model.train()
    running_train_loss = 0.0
    
    for batch_idx, (images, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item() * images.size(0)
        
        # Print progress every 15 batches so you know it's moving
        if (batch_idx + 1) % 15 == 0 or (batch_idx + 1) == len(train_loader):
            print(f"Epoch [{epoch}/{num_epochs}] | Batch [{batch_idx + 1}/{len(train_loader)}] | Current Batch Loss: {loss.item():.4f}")
            
    train_loss = running_train_loss / len(train_loader.dataset)
    
    # ------------------
    # Validation Phase
    # ------------------
    model.eval()
    running_val_loss = 0.0
    
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            running_val_loss += loss.item() * images.size(0)
            
    val_loss = running_val_loss / len(val_loader.dataset)
    elapsed = time.time() - start_time
    
    print("-" * 60)
    print(f"Epoch [{epoch}/{num_epochs}] Complete ({elapsed:.1f}s) | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    # Save the best model checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "models/unet.pth")
        print(f"--> Saved best model checkpoint to 'models/unet.pth'")
    print("-" * 60)

print("\nTraining completed successfully! Model saved at models/unet.pth")